# MCTS on Catan — Analysis Notebook

Read-only over `mcts_study/runs/`. Produces the plots that go in `docs/writeup.md`.

**Note on scale.** Production runs in this repo are deliberately small (3-50 games per cell, sims grids capped at 100-400) because Phase-0 chance points blow Tier-1 game length up to ~12-15k steps — at the spec's original sims=4000 with 200 games/cell, one experiment would take days. See `docs/writeup.md` §6 (Caveats) and the per-experiment `config.json` in each run directory for actual parameters.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

RUNS = Path('../runs')

def latest_run(name_prefix: str) -> Path:
    """Return the most recent run dir matching `*-name_prefix`."""
    candidates = sorted(RUNS.glob(f'*-{name_prefix}'), key=lambda p: p.name)
    if not candidates:
        raise FileNotFoundError(f'no runs found under {RUNS} matching *-{name_prefix}')
    return candidates[-1]

def load_run(run_dir: Path):
    games = pq.read_table(run_dir / 'games.parquet').to_pandas()
    moves = pq.read_table(run_dir / 'moves.parquet').to_pandas()
    config = json.loads((run_dir / 'config.json').read_text())
    return games, moves, config

## Experiment 1 — Win-rate vs simulation budget

MCTS player at seat 0 vs 3 RandomBots. Sweep `max_simulations` and measure the MCTS player's win-rate.

In [ ]:
e1_dir = latest_run('e1_winrate_vs_random')
e1_games, e1_moves, e1_cfg = load_run(e1_dir)
print(f'Loaded {e1_dir.name}: {len(e1_games)} games, sims grid = {e1_cfg.get("sims_per_move_grid")}')

# Each game's seed encodes the sims used: seed = seed_base + sims*1000 + i
seed_base = e1_cfg.get('seed_base', 1_000_000)
e1_games['sims'] = ((e1_games['seed'] - seed_base) // 1000)

winrates = e1_games.groupby('sims').apply(lambda g: (g['winner'] == 0).mean()).reset_index(name='win_rate_seat0')
n_per_cell = e1_games.groupby('sims').size().reset_index(name='n')
summary = winrates.merge(n_per_cell, on='sims')
print(summary)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(summary['sims'], summary['win_rate_seat0'], 'o-')
ax.axhline(0.25, ls='--', color='gray', label='random baseline (1/4)')
ax.set_xlabel('max_simulations')
ax.set_ylabel('MCTS player win-rate')
ax.set_title('e1: MCTS vs 3 RandomBots, win-rate scaling with simulation budget')
ax.legend()
plt.tight_layout(); plt.show()

## Experiment 2 — UCB exploration constant `c` sweep

In [ ]:
try:
    e2_dir = latest_run('e2_ucb_c_sweep')
    e2_games, _, e2_cfg = load_run(e2_dir)
    print(f'Loaded {e2_dir.name}: {len(e2_games)} games')

    # seed = seed_base + int(c*100)*1_000 + i
    seed_base2 = e2_cfg.get('seed_base', 2_000_000)
    e2_games['c_x100'] = ((e2_games['seed'] - seed_base2) // 1000) % 1000
    e2_games['c'] = e2_games['c_x100'] / 100

    by_c = e2_games.groupby('c').apply(lambda g: (g['winner'] == 0).mean()).reset_index(name='win_rate')
    print(by_c)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(by_c['c'], by_c['win_rate'], 'o-')
    ax.axhline(0.25, ls='--', color='gray', label='random baseline')
    ax.set_xlabel('UCB exploration constant c')
    ax.set_ylabel('MCTS player win-rate')
    ax.set_title(f'e2: c sweep at sims={e2_cfg.get("sims_per_move")}')
    ax.legend()
    plt.tight_layout(); plt.show()
except FileNotFoundError as e:
    print('e2 not run yet:', e)

## Experiment 3 — Random vs heuristic rollouts

In [ ]:
try:
    e3_dir = latest_run('e3_rollout_policy')
    e3_games, _, e3_cfg = load_run(e3_dir)
    print(f'Loaded {e3_dir.name}: {len(e3_games)} games')

    # seed = seed_base + (0 random | 1 heuristic) * 1_000_000 + sims*1000 + i
    seed_base3 = e3_cfg.get('seed_base', 3_000_000)
    rel = e3_games['seed'] - seed_base3
    e3_games['policy'] = np.where(rel >= 1_000_000, 'heuristic', 'random')
    e3_games['sims'] = ((rel % 1_000_000) // 1000)

    by_grp = e3_games.groupby(['policy', 'sims']).apply(lambda g: (g['winner'] == 0).mean()).reset_index(name='win_rate')
    print(by_grp)

    fig, ax = plt.subplots(figsize=(6, 4))
    for policy, sub in by_grp.groupby('policy'):
        ax.semilogx(sub['sims'], sub['win_rate'], 'o-', label=policy)
    ax.axhline(0.25, ls='--', color='gray', label='random baseline')
    ax.set_xlabel('max_simulations')
    ax.set_ylabel('MCTS player win-rate')
    ax.set_title('e3: rollout policy comparison')
    ax.legend()
    plt.tight_layout(); plt.show()
except FileNotFoundError as e:
    print('e3 not run yet:', e)

## Experiment 4 — Tournament (MCTS vs Greedy vs Random)

In [ ]:
try:
    e4_dir = latest_run('e4_tournament')
    e4_games, _, e4_cfg = load_run(e4_dir)
    print(f'Loaded {e4_dir.name}: {len(e4_games)} games')

    # seed = seed_base + rot_idx * 10_000 + i — rot_idx 0..3 maps to which seat MCTS sits in.
    seed_base4 = e4_cfg.get('seed_base', 4_000_000)
    e4_games['rotation'] = ((e4_games['seed'] - seed_base4) // 10_000)

    # Across rotations, what's the win-rate of the MCTS / Greedy / Random *roles*?
    # rotation r places MCTS at seat r, Greedy at seat (r+1)%4, Random at seats (r+2,r+3)%4.
    def role_at_seat(seat: int, rotation: int) -> str:
        rel = (seat - rotation) % 4
        return ['MCTS', 'Greedy', 'Random', 'Random'][rel]

    role_wins = {'MCTS': 0, 'Greedy': 0, 'Random': 0}
    role_seats = {'MCTS': 0, 'Greedy': 0, 'Random': 0}
    for _, row in e4_games.iterrows():
        rot = row['rotation']
        for seat in range(4):
            r = role_at_seat(seat, rot)
            role_seats[r] += 1
            if row['winner'] == seat:
                role_wins[r] += 1

    role_wr = {r: role_wins[r] / role_seats[r] for r in role_wins}
    print('Per-role win rates (averaged over rotations):', role_wr)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(role_wr.keys(), role_wr.values())
    ax.axhline(0.25, ls='--', color='gray', label='random baseline (1/4)')
    ax.set_ylabel('Per-seat win-rate (avg over rotations)')
    ax.set_title(f'e4: tournament at mcts_sims={e4_cfg.get("mcts_sims")}')
    ax.legend()
    plt.tight_layout(); plt.show()
except FileNotFoundError as e:
    print('e4 not run yet:', e)

## Discussion prompts (for `docs/writeup.md`)

Things to interpret once the plots are live:
- e1: what's the shape of the curve? plateau? where? does the gap to the random-baseline open monotonically with sims?
- e2: best `c` empirically vs theoretical `c=√2≈1.414`. Sharp peak or wide plateau?
- e3: classical lesson — heuristic rollouts win at low budgets, lose at high budgets? Did we see the crossover?
- e4: rank-ordering check. Does MCTS > Greedy > Random hold? Where do the gaps come from?
- Sample sizes: at our small num_games per cell, what's the noise floor on these win-rates?